## Aplicación simultánea de entropía y coeficiente de Gini

### Preprocesamiento de DataFrame

In [3]:
import sys
from pathlib import Path
import numpy as np
import polars as pl

#Para poder utilizar archivos del proyecto
ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.cleaning.preprocessing import pipeline_preprocesamiento

df = pipeline_preprocesamiento()

### Cálculo de medidas

In [ ]:
# Ordenar de forma ascendente por frecuencia y calcular proporciones
frecuencias = (
    df.group_by("estrato_socioeconomico")
    .agg(n=pl.len())
    .sort("n")
    .with_columns(p=pl.col("n") / pl.col("n").sum())
)

K = frecuencias.height
N = frecuencias["n"].sum()

# Entropía de Shannon
p_array = frecuencias["p"].to_numpy()
shannon_bits = -np.sum(p_array * np.log2(p_array))
shannon_max = np.log2(K)
shannon_rel = shannon_bits/shannon_max

# Coeficiente de Gini sobre las frecuencias
x = frecuencias["n"].to_numpy()
i = np.arange(1, K + 1)
sum_x = np.sum(x)
sum_ix = np.sum(i * x)

gini_frecuencias = (2 * sum_ix - (K + 1) * sum_x) / (K * sum_x)

# Tabla
resumen_estrato = pl.DataFrame(
    {
        "Variable": ["estrato_socioeconomico"],
        "Categorías (K)": [K],
        "Entropía Shannon (bits)": [round(shannon_bits, 4)],
        "Entropía Máxima (bits)": [round(shannon_max, 4)],
        "Entropía Relativa": [f"{shannon_rel * 100:.2f}%"],
        "Gini Frecuencias": [round(gini_frecuencias, 4)],
    }
)

print(resumen_estrato)

shape: (1, 6)
┌────────────────────┬───────────────────┬───────────┬───────────────────┬───────────┬─────────────┐
│ Variable           ┆ Categorías (K)    ┆ Entropía  ┆ Entropía Máxima   ┆ Entropía  ┆ Gini        │
│ ---                ┆ ---               ┆ Shannon   ┆ (bits)            ┆ Relativa  ┆ Frecuencias │
│ str                ┆ i64               ┆ (bits)    ┆ ---               ┆ ---       ┆ ---         │
│                    ┆                   ┆ ---       ┆ f64               ┆ str       ┆ f64         │
│                    ┆                   ┆ f64       ┆                   ┆           ┆             │
╞════════════════════╪═══════════════════╪═══════════╪═══════════════════╪═══════════╪═════════════╡
│ estrato_socioecono ┆ 4                 ┆ 1.7169    ┆ 2.0               ┆ 85.84%    ┆ 0.3294      │
│ mico               ┆                   ┆           ┆                   ┆           ┆             │
└────────────────────┴───────────────────┴───────────┴───────────────────┴───